asd

In [6]:
import pandas as pd
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np


dtype={'index':int,'submission_id':str,'title':str,'selftext':str,'num_comments':int,'num_unique_commentators':int,'ups':int,'upvote_ratio':float,'author':str,'created_utc':str,'text_only':str}
showerthoughts_data = pd.read_csv('reddit_showerthoughts.tsv',sep='\t',encoding="utf-8",on_bad_lines='warn',)
showerthoughts_data.shape

C:\Users\z5612172\AppData\Local\Temp\ipykernel_20860\2414998430.py:8: ParserWarning: Skipping line 40357: expected 11 fields, saw 12

  showerthoughts_data = pd.read_csv('reddit_showerthoughts.tsv',sep='\t',encoding="utf-8",on_bad_lines='warn',)


(95086, 11)

In [ ]:
posts=showerthoughts_data
# print(posts.dtypes)
print(f"Loaded {len(posts)} sample submissions.")
# We only need the ID and title for this task.
# posts=posts.rename(columns={'SUBMISSION_ID':"id",'TITLE':'title','SUBREDDIT':'subreddit','AUTHOR':'author'})
posts['title']=posts['title'].replace('[deleted by user]',np.nan)

posts = posts.dropna(subset=['title'])
posts.shape

posts['selftext']=posts['selftext'].fillna('').replace('[removed]','')
posts['selftext']=posts['selftext'].replace('[deleted]','')
posts['combined']=posts['title'].str.replace(r'\.$', '', regex=True)+'. '+posts['selftext']

posts[30120:30130]
print(f"Processing {len(posts)} posts after dropping ones with no title.")


# --- 2. Initialize Model and Generate Embeddings ---
# Load a pre-trained model from sentence-transformers.
# 'all-MiniLM-L6-v2' is a good, fast model for general purpose use.
print("Loading the sentence transformer model...")
# model = SentenceTransformer('all-MiniLM-L6-v2')
model = SentenceTransformer('all-mpnet-base-v2')


# Prepare the sentences for the model.
# The model expects a list of strings.
# post_titles = list(map(str,posts['title'].tolist()))
# post_selftext = list(map(str,posts['selftext'].tolist()))
post_combineds = list(map(str,posts['combined'].tolist()))


print("Generating embeddings for post combineds... (This may take a few minutes)")
# title_embeddings = model.encode(post_titles, show_progress_bar=True)
title_embeddings = model.encode(post_combineds, show_progress_bar=True)

print("Embeddings generated successfully!")
print(f"Shape of the embeddings array: {title_embeddings.shape}") # (num_posts, embedding_dimension)

# print("Generating embeddings for post content... (This may take a few minutes)")
# content_embeddings= model.encode(post_selftext, show_progress_bar=True)
# print("Embeddings generated successfully!")
# print(f"Shape of the embeddings array: {content_embeddings.shape}") # (num_posts, embedding_dimension)


# --- 3. Save the Results ---
# It's essential to save your embeddings so you don't have to regenerate them.
# We'll save the post IDs and their corresponding embeddings.
# post_ids = posts['index'].tolist()
post_ids = posts['submission_id'].tolist()


# Use numpy's savez_compressed to save multiple arrays efficiently.
# np.savez_compressed(
#     'post_embeddings.npz',
#     ids=post_ids,
#     title_embed=title_embeddings,
#     content_embed=content_embeddings

# )
np.savez_compressed(
    'showerthoughts_combined_embeddings.npz',
    ids=post_ids,
    combined_embed=title_embeddings,
    # content_embed=content_embeddings

)

print("\nSuccessfully saved post IDs and embeddings to 'post_embeddings.npz'.")
print("You can now load this file in another script to build your recommender.")

Loaded 95086 sample submissions.


C:\Users\z5612172\AppData\Local\Temp\ipykernel_20860\1737863871.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  posts['selftext']=posts['selftext'].fillna('').replace('[removed]','')
C:\Users\z5612172\AppData\Local\Temp\ipykernel_20860\1737863871.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  posts['selftext']=posts['selftext'].replace('[deleted]','')
C:\Users\z5612172\AppData\Local\Temp\ipykernel_20860\1737863871.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice fr

,index,submission_id,title,selftext,num_comments,num_unique_commentators,ups,upvote_ratio,author,created_utc,text_only,combined
31115,31115,t3_amp76q,Mike Wazowski's chin is also his crotch,,0,0,16671,0.92,Guv83,1.549197e+09,True,Mike Wazowski's chin is also his crotch.
31116,31116,t3_9aqf52,Human is the only animal that use tools to pre...,,0,0,5,0.67,kingrammus12,1.535387e+09,True,Human is the only animal that use tools to pre...
31117,31117,t3_9vm74y,It's not the friendzone if you actually value ...,,0,0,127,0.92,YaBig_Dingus,1.541783e+09,True,It's not the friendzone if you actually value ...
31118,31118,t3_84ozyh,Even though Stephen Hawking is gone from this ...,,0,0,10806,0.95,Nobody_home,1.521140e+09,True,Even though Stephen Hawking is gone from this ...
31119,31119,t3_c5s1lz,"Holy shit, shit has hit in it.",,0,0,0,0.44,digitalenabler,1.561567e+09,True,"Holy shit, shit has hit in it."
31120,31120,t3_eyjde5,Auntie Annes needs to partner with the movie t...,,0,0,3930,0.91,NaN,1.580786e+09,True,Auntie Annes needs to partner with the movie t...
31121,31121,t3_au9lao,"A cable tie (zip tie) is never too short, if y...",Hit me today as I connected 3 ties together to...,0,0,10,0.82,Troby01,1.551026e+09,True,"A cable tie (zip tie) is never too short, if y..."
31122,31122,t3_f4iybx,Calling certain food Chinese food or Korean fo...,,0,0,1,0.52,O0OO0OF,1.581815e+09,True,Calling certain food Chinese food or Korean fo...
31123,31123,t3_9if7fb,"The nicer a vehicle or piece of furniture is, ...",Example: at your grandma's house there's that ...,0,0,10,0.85,turtledragon27,1.537765e+09,True,"The nicer a vehicle or piece of furniture is, ..."
31124,31124,t3_cj5mzk,"Maybe muppets have such large, bulging eyes be...",,0,0,10252,0.92,JoeFas,1.564370e+09,True,"Maybe muppets have such large, bulging eyes be..."


In [8]:
posts.head(10)

,index,submission_id,title,selftext,num_comments,num_unique_commentators,ups,upvote_ratio,author,created_utc,text_only,combined
0,0,t3_ea85qk,You might have seen a random person two times ...,NaN,0,0,22,0.83,Your-Mom-Gabe,1.576263e+09,True,NaN
1,1,t3_201knk,"If you have two arms, then you have the above ...",Because just one person with one arm would thr...,0,0,28,0.71,NaN,1.394456e+09,True,"If you have two arms, then you have the above ..."
2,2,t3_7wa7jw,"Statistically speaking, you have a greater cha...","Crazy, huh?",0,0,3,0.62,420TreeExpert,1.518142e+09,True,"Statistically speaking, you have a greater cha..."
3,3,t3_cnyfjq,Gay genes are transferred to the unborn baby w...,NaN,0,0,0,0.27,NaN,1.565332e+09,True,NaN
4,4,t3_c1bgf9,A woodpecker has to have headaches,NaN,0,0,3,0.59,budshady04,1.560701e+09,True,NaN
5,5,t3_dv1u7e,Jake from adventure Time could make his dick a...,NaN,0,0,108,0.92,islamicsuicidebomber,1.573518e+09,True,NaN
6,6,t3_3a4puu,When I die I want to be buried at a fat camp s...,NaN,0,0,1388,0.87,germattack3,1.434518e+09,True,NaN
7,7,t3_d3jil3,"When vibrations in the air hit your eardrum, o...",NaN,0,0,53,0.77,douggold11,1.568347e+09,True,NaN
8,8,t3_bv6yox,"If we survive, there may eventually be a compa...",NaN,0,0,7,0.82,Sigh_SMH,1.559306e+09,True,NaN
9,9,t3_8hzlif,If a couple of people are asses to you they ar...,NaN,0,0,38,0.79,JuGGrNauT_,1.525808e+09,True,NaN
